# Simulate a small company

a) Connect python to gemini, very important that you place the api key in .env and gitignore it

In [1]:
# Connect to Gemini and load API key

from dotenv import load_dotenv
import google.generativeai as genai
import os


load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY could not be found in the .env-file")

# configure Gemoni
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-1.5-flash')

print("Setup complete. Connection to Gemini was successfull")

Setup complete. Connection to Gemini was successfull


b)

In [2]:
employee_prompt = """
Generate a list with 20 employees for a small swedish tech-company.
The response should ONLY be in a JSON-array of objects, without any explanations, text, markdown-format or anything.

Each object in the array must include the following fields:
- "first_name": A regular swedish first name.
- "last_name": A regular swedish last name.
- "phone_number": A swedish phone number in the format '+46 7X XXX XX XX'.
- "email": A unique email adress based on the company name, with the domain 'denthelp.se'.
- "department": Must be one of the following: "IT", "HR", "Marketing", "Sales".
- "salary": A reasonable monthly pay in swedish kronor(SEK) for the department the person is working in, between 35000SEK and 125000SEK.
- "title": A fitting jobtitle for the department the person is working in (for example 'Software Engineer' for it, 'Sales Manager' for sales).

"""


print("Sending prompt to Gemini...")
response = model.generate_content(employee_prompt)
raw_employee_data = response.text

print("Raw data returned from Gemini")
print(raw_employee_data)

Sending prompt to Gemini...
Raw data returned from Gemini
```json
[
  {"first_name": "Anna", "last_name": "Andersson", "phone_number": "+46 70 123 45 67", "email": "anna.andersson@denthelp.se", "department": "IT", "salary": 65000, "title": "Software Engineer"},
  {"first_name": "Lars", "last_name": "Eriksson", "phone_number": "+46 73 147 85 20", "email": "lars.eriksson@denthelp.se", "department": "Sales", "salary": 50000, "title": "Sales Representative"},
  {"first_name": "Maria", "last_name": "Johansson", "phone_number": "+46 70 987 65 43", "email": "maria.johansson@denthelp.se", "department": "Marketing", "salary": 55000, "title": "Marketing Coordinator"},
  {"first_name": "Per", "last_name": "Nilsson", "phone_number": "+46 72 555 12 12", "email": "per.nilsson@denthelp.se", "department": "IT", "salary": 75000, "title": "Data Analyst"},
  {"first_name": "Eva", "last_name": "Svensson", "phone_number": "+46 70 333 44 55", "email": "eva.svensson@denthelp.se", "department": "HR", "salary"

c)

In [3]:
from pydantic import BaseModel, Field, EmailStr
from typing import Literal

class Employee(BaseModel):
    first_name: str
    last_name: str
    phone_number: str = Field(pattern=r"^\+46\s7[0-9]\s[0-9]{3}\s[0-9]{2}\s[0-9]{2}$")
    email: EmailStr
    department: Literal["IT", "HR", "Marketing", "Sales"]
    salary: int = Field(gt=25000, lt=150000)
    title: str

def clean_json_string(raw_text: str) -> str:
    start_index = raw_text.rfind('[')
    end_index = raw_text.rfind(']')
    if start_index != -1 and end_index != -1:
        return raw_text[start_index : end_index + 1]
    
print("Cleaning and validating the data...")
cleaned_data_str = clean_json_string(raw_employee_data)
validated_employees = []
invalid_count = 0

Cleaning and validating the data...


In [4]:
import json
from pydantic import ValidationError
if cleaned_data_str:
    try:
        employee_list_from_json = json.loads(cleaned_data_str)
        for employee_data in employee_list_from_json:
            try:
                validated_employee = Employee(**employee_data)
                validated_employees.append(validated_employee)
            except ValidationError as e:
                invalid_count += 1
                print(f"\n--- Nonvalid post archived ---\nData: {employee_data}\nError: {e}\n------------------")
        print(f"Validation done! {len(validated_employees)} got approved, {invalid_count} was archived.")
    except json.JSONDecodeError:
        print("Error: Could not validate the text as a JSON after cleaning.")
else:
    print("Error: No JSON-array was found in the raw data.")    


Validation done! 20 got approved, 0 was archived.
